# DGUFS — Basic Usage

This notebook demonstrates the **Dependence Guided Unsupervised Feature Selection (DGUFS)** algorithm on the Iris dataset.

DGUFS selects the most informative features from unlabelled data by jointly optimising a feature selection matrix and a graph-based clustering structure via ADMM.

**Reference:** Jun Guo and Wenwu Zhu, *Dependence Guided Unsupervised Feature Selection*, AAAI 2018.

## 1. Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

from dgufs import DGUFS
from dgufs.utils import normalized_mutual_information

## 2. Load and preprocess data

We use the Iris dataset (150 samples, 4 features, 3 classes). DGUFS is unsupervised — labels are only used at the end to evaluate clustering quality.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names = iris.target_names

print(f"Dataset shape : {X.shape}")
print(f"Features      : {feature_names}")
print(f"Classes       : {class_names}")

# Standardise — DGUFS uses pairwise distances internally
X_std = StandardScaler().fit_transform(X)
print(f"\nStandardised mean  : {X_std.mean(axis=0).round(6)}")
print(f"Standardised std   : {X_std.std(axis=0).round(6)}")

## 3. Fit DGUFS

Key hyperparameters:
| Parameter | Meaning |
|---|---|
| `num_features` | How many features to keep |
| `num_clusters` | Expected number of clusters (matches the rank constraint on the graph matrix L) |
| `alpha` | Controls rank of the graph matrix (higher → sparser graph) |
| `beta` | Balances the similarity term vs. the feature-dependence term |

In [ ]:
selector = DGUFS(num_features=2, num_clusters=3, alpha=0.5, beta=0.9, max_iter=100)
selector.fit(X_std)

print("Fitting complete.")
print(f"Selected feature indices : {selector.support}")
print(f"Selected feature names   : {[feature_names[i] for i in selector.support]}")

## 4. Transform the data

`transform()` returns a matrix with only the selected features, preserving the original type (numpy array or pandas DataFrame).

In [ ]:
X_reduced = selector.transform(X_std)

print(f"Original shape : {X_std.shape}")
print(f"Reduced shape  : {X_reduced.shape}")

## 5. Cluster memberships

DGUFS simultaneously learns a graph structure that encodes cluster assignments. The `memberships` property returns a label per sample derived from the graph's leading eigenvectors.

In [ ]:
cluster_labels = selector.memberships

print(f"Cluster labels (first 20): {cluster_labels[:20]}")
print(f"Unique clusters found    : {np.unique(cluster_labels)}")

nmi = normalized_mutual_information(y, cluster_labels)
print(f"\nNormalized Mutual Information vs. true labels: {nmi:.4f}")
print("(NMI = 1.0 is a perfect match; 0.0 is no agreement)")

## 6. Visualise results

Two scatter plots side by side:
- **Left**: samples coloured by DGUFS cluster assignment
- **Right**: samples coloured by true Iris class label

In [ ]:
selected_names = [feature_names[i] for i in selector.support]
x_col, y_col = X_reduced[:, 0], X_reduced[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- DGUFS cluster assignments ---
scatter0 = axes[0].scatter(x_col, y_col, c=cluster_labels, cmap="tab10", s=30, alpha=0.8)
axes[0].set_title("DGUFS cluster assignments")
axes[0].set_xlabel(selected_names[0])
axes[0].set_ylabel(selected_names[1])
legend0 = axes[0].legend(
    *scatter0.legend_elements(),
    title="Cluster",
    loc="upper left",
)
axes[0].add_artist(legend0)

# --- True class labels ---
scatter1 = axes[1].scatter(x_col, y_col, c=y, cmap="tab10", s=30, alpha=0.8)
axes[1].set_title("True Iris class labels")
axes[1].set_xlabel(selected_names[0])
axes[1].set_ylabel(selected_names[1])
axes[1].legend(
    handles=scatter1.legend_elements()[0],
    labels=list(class_names),
    title="Class",
    loc="upper left",
)

fig.suptitle(
    f"Selected features: {selected_names}  |  NMI = {nmi:.4f}",
    fontsize=13,
    y=1.02,
)
plt.tight_layout()
plt.show()

## 7. Using DGUFS in a scikit-learn Pipeline

Because `DGUFS` inherits from `BaseEstimator` and `TransformerMixin`, it slots directly into a scikit-learn pipeline.

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", DGUFS(num_features=2, num_clusters=3)),
])

X_pipeline = pipeline.fit_transform(X)
print(f"Pipeline output shape: {X_pipeline.shape}")